In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("heart-eda") \
    .config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow") \
    .config("spark.executor.extraJavaOptions", "-Djava.security.manager=allow") \
    .getOrCreate()

In [2]:
df = spark.read.csv("../../media/datasets/heart_disease.csv", header=True, inferSchema=True)
df.show(5)

+----+------+--------------+-----------------+---------------+-------+--------------------+--------+------------------+-------------------+-------------------+--------------------+-------------------+------------+-----------------+-----------------+------------------+-------------------+------------------+------------------+--------------------+
| Age|Gender|Blood Pressure|Cholesterol Level|Exercise Habits|Smoking|Family Heart Disease|Diabetes|               BMI|High Blood Pressure|Low HDL Cholesterol|High LDL Cholesterol|Alcohol Consumption|Stress Level|      Sleep Hours|Sugar Consumption|Triglyceride Level|Fasting Blood Sugar|         CRP Level|Homocysteine Level|Heart Disease Status|
+----+------+--------------+-----------------+---------------+-------+--------------------+--------+------------------+-------------------+-------------------+--------------------+-------------------+------------+-----------------+-----------------+------------------+-------------------+----------------

In [3]:
df.printSchema()

root
 |-- Age: double (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Blood Pressure: double (nullable = true)
 |-- Cholesterol Level: double (nullable = true)
 |-- Exercise Habits: string (nullable = true)
 |-- Smoking: string (nullable = true)
 |-- Family Heart Disease: string (nullable = true)
 |-- Diabetes: string (nullable = true)
 |-- BMI: double (nullable = true)
 |-- High Blood Pressure: string (nullable = true)
 |-- Low HDL Cholesterol: string (nullable = true)
 |-- High LDL Cholesterol: string (nullable = true)
 |-- Alcohol Consumption: string (nullable = true)
 |-- Stress Level: string (nullable = true)
 |-- Sleep Hours: double (nullable = true)
 |-- Sugar Consumption: string (nullable = true)
 |-- Triglyceride Level: double (nullable = true)
 |-- Fasting Blood Sugar: double (nullable = true)
 |-- CRP Level: double (nullable = true)
 |-- Homocysteine Level: double (nullable = true)
 |-- Heart Disease Status: string (nullable = true)



In [4]:
df.count()

10000

In [5]:
from pyspark.sql.functions import col, sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+---+------+--------------+-----------------+---------------+-------+--------------------+--------+---+-------------------+-------------------+--------------------+-------------------+------------+-----------+-----------------+------------------+-------------------+---------+------------------+--------------------+
|Age|Gender|Blood Pressure|Cholesterol Level|Exercise Habits|Smoking|Family Heart Disease|Diabetes|BMI|High Blood Pressure|Low HDL Cholesterol|High LDL Cholesterol|Alcohol Consumption|Stress Level|Sleep Hours|Sugar Consumption|Triglyceride Level|Fasting Blood Sugar|CRP Level|Homocysteine Level|Heart Disease Status|
+---+------+--------------+-----------------+---------------+-------+--------------------+--------+---+-------------------+-------------------+--------------------+-------------------+------------+-----------+-----------------+------------------+-------------------+---------+------------------+--------------------+
| 29|    19|            19|               30|    

In [6]:
df.groupBy("Heart Disease Status").count().show()

+--------------------+-----+
|Heart Disease Status|count|
+--------------------+-----+
|                  No| 8000|
|                 Yes| 2000|
+--------------------+-----+



In [7]:
from pyspark.sql.functions import avg

df.groupBy("Gender").agg(
    avg((col("Heart Disease Status") == "Yes").cast("int")).alias("disease_rate")
).show()

+------+-------------------+
|Gender|       disease_rate|
+------+-------------------+
|  NULL|0.05263157894736842|
|Female|  0.206910405785456|
|  Male| 0.1936837897261643|
+------+-------------------+



In [8]:
df.select("Age").describe().show()

+-------+------------------+
|summary|               Age|
+-------+------------------+
|  count|              9971|
|   mean|49.296259151539466|
| stddev|18.193970063916655|
|    min|              18.0|
|    max|              80.0|
+-------+------------------+



In [11]:
df.filter(col("Cholesterol Level") > 200).show()

+----+------+--------------+-----------------+---------------+-------+--------------------+--------+------------------+-------------------+-------------------+--------------------+-------------------+------------+-----------------+-----------------+------------------+-------------------+--------------------+------------------+--------------------+
| Age|Gender|Blood Pressure|Cholesterol Level|Exercise Habits|Smoking|Family Heart Disease|Diabetes|               BMI|High Blood Pressure|Low HDL Cholesterol|High LDL Cholesterol|Alcohol Consumption|Stress Level|      Sleep Hours|Sugar Consumption|Triglyceride Level|Fasting Blood Sugar|           CRP Level|Homocysteine Level|Heart Disease Status|
+----+------+--------------+-----------------+---------------+-------+--------------------+--------+------------------+-------------------+-------------------+--------------------+-------------------+------------+-----------------+-----------------+------------------+-------------------+------------

In [12]:
df.groupBy("Smoking").count().show()
df.groupBy("Exercise Habits").count().show()
df.groupBy("Stress Level").count().show()

+-------+-----+
|Smoking|count|
+-------+-----+
|   NULL|   25|
|     No| 4852|
|    Yes| 5123|
+-------+-----+

+---------------+-----+
|Exercise Habits|count|
+---------------+-----+
|           High| 3372|
|            Low| 3271|
|           NULL|   25|
|         Medium| 3332|
+---------------+-----+

+------------+-----+
|Stress Level|count|
+------------+-----+
|        High| 3271|
|         Low| 3320|
|        NULL|   22|
|      Medium| 3387|
+------------+-----+

